# Apt 305 — ISO 52016-1 engine changes, and ISO vs EnergyPlus

Runs the same Melbourne apartment through three engine versions, then runs the **unmodified**
engine against EnergyPlus 24.1 on identical inputs.

| Branch | Engine |
|---|---|
| `claude/pybuildingenergy-baseline-anjro8` | Unmodified ISO 52016-1 |
| `claude/dynamic-window-properties-anjro8` | + dynamic window properties (`U_win(t)`, angular `g_win`) |
| `claude/window-plus-dynamic-hce-anjro8` | + wind-dependent surface heat transfer coefficients |

Each branch runs in its own subprocess — three versions of the same `pybuildingenergy`
package cannot share one `sys.path`.

### One weather file, everywhere

Every run below is pinned to the **same** EPW, committed in `weather_cache/`.

That is deliberate. These two scripts previously disagreed — 150 kWh heating in one
chart against 40 kWh in the other, on what was supposedly the same engine and the same
building — because one run had silently fallen back to a PVGIS TMY while the other used
an EPW. Same engine, same building, *different weather*.

Both scripts now write `run_meta.json` recording which file they actually used, and the
last cell asserts the two baselines match to 1e-6.

**Runtime:** roughly 15–20 minutes end to end (the EnergyPlus download and four annual
simulations dominate).

> **Private repo?** If the clone below asks for credentials, create a GitHub personal
> access token with `repo` scope and use the commented-out line instead.

In [ ]:
# 1. Clone the repository (all branches — the comparison needs them)
REPO = "https://github.com/samiraghafarigousheh-sys/AIB.git"
BRANCH = "claude/window-plus-dynamic-hce-anjro8"

# Private repo? Uncomment and paste a token with `repo` scope:
# TOKEN = "ghp_xxx"
# REPO = f"https://{TOKEN}@github.com/samiraghafarigousheh-sys/AIB.git"

import os, shutil
if os.path.isdir("AIB"):
    shutil.rmtree("AIB")

!git clone --quiet --branch $BRANCH $REPO AIB
%cd AIB

# Make sure every branch is present locally, so `git worktree` can reach them.
!git fetch --quiet origin '+refs/heads/*:refs/remotes/origin/*'
!git branch -r

In [ ]:
# 2. Dependencies
!pip install -q -r pybuildingenergy/requirements.txt
!pip install -q plotly            # interactive Sankey; the PNG works without it

# git needs an identity before `worktree add` will run in a fresh container
!git config user.email "colab@example.com"
!git config user.name  "Colab"
print("dependencies installed")

In [ ]:
# 3. Pin the weather file — ONE file for every run in this notebook.
#
# The EPW ships in the repo, so nothing is downloaded and no run can quietly
# fall back to a different source. EnergyPlus can only read an EPW anyway.
import glob, sys
sys.path.insert(0, "examples")

EPW = sorted(glob.glob("weather_cache/*.epw"))[0]

from weather_melbourne import read_epw_site, site_offset_deg
lat, lon, city = read_epw_site(EPW)
print(f"weather file : {EPW}")
print(f"station      : {city}  (lat {lat}, lon {lon})")
print(f"offset from central Melbourne: {site_offset_deg(lat, lon):.2f} deg")

# To use a different station, upload it and re-run this cell:
#   from google.colab import files; files.upload()
#   !mv *.epw weather_cache/
# Sources: https://climate.onebuilding.org
#   -> WMO Region 5 > AUS_Australia > VIC_Victoria

In [ ]:
# 4. Three engine branches on that EPW (slow cell — three annual simulations)
!python examples/compare_branches_apt305.py \
    --weather "$EPW" --require-epw --outdir results/apt305

In [ ]:
# 5. Table + chart: effect of each engine change
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv("results/apt305/comparison.csv"))
display(Image("results/apt305/apt305_comparison.png"))

---
## Baseline ISO 52016-1 vs EnergyPlus

Same building, same weather, same schedules — the *unmodified* engine against
EnergyPlus 24.1. None of the AIB changes are involved here.

The audit runs first: it lists every parameter that affects the result and marks each
ALIGNED / FIXED / METHOD. Several ISO behaviours are invisible in the building dictionary.

In [ ]:
# 6. Install EnergyPlus 24.1
EP_URL = ("https://github.com/NREL/EnergyPlus/releases/download/v24.1.0/"
          "EnergyPlus-24.1.0-9d7789a3ac-Linux-Ubuntu22.04-x86_64.tar.gz")
!wget -q -O /tmp/ep.tar.gz $EP_URL
!mkdir -p /opt/ep && tar -xzf /tmp/ep.tar.gz -C /opt/ep --strip-components=1
!/opt/ep/energyplus --version

In [ ]:
# 7. Parameter alignment audit (no simulation)
!python examples/baseline_vs_energyplus.py --audit-only

In [ ]:
# 8. Both engines — on the SAME EPW as cell 4
!python examples/baseline_vs_energyplus.py \
    --energyplus /opt/ep/energyplus \
    --weather "$EPW" \
    --outdir results/baseline_vs_ep

In [ ]:
# 9. Heating / cooling / total — table, bar chart, and the ISO energy-balance Sankey
import pandas as pd
from IPython.display import Image, display, Markdown

display(Markdown("### Heating, cooling and total — ISO 52016-1 vs EnergyPlus"))
display(pd.read_csv("results/baseline_vs_ep/baseline_vs_energyplus.csv"))
display(Image("results/baseline_vs_ep/baseline_vs_energyplus.png"))

display(Markdown("### pyBuildingEnergy annual energy balance (Sankey)"))
display(Image("results/baseline_vs_ep/sankey_pybuildingenergy.png"))

In [ ]:
# 10. Interactive Sankey.
#
# Rendered from a figure object, NOT from the .html file on disk. Pointing an
# IFrame at a local path is what produced "localhost refused to connect" —
# Colab serves no web server on that path. The PNG above always works; this is
# the hoverable version.
import json, sys
sys.path.insert(0, "examples")
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "colab"

from baseline_vs_energyplus import sankey_flows, SANKEY_IN_COLORS, SANKEY_OUT_COLORS

iso = json.load(open("results/baseline_vs_ep/iso_results.json"))
inflows, outflows = sankey_flows(iso)

labels = [n for n, _ in inflows] + ["Zone"] + [n for n, _ in outflows]
z = len(inflows)
colors = ([SANKEY_IN_COLORS[i] for i in range(len(inflows))] + ["#52514e"]
          + [SANKEY_OUT_COLORS[i] for i in range(len(outflows))])

fig = go.Figure(go.Sankey(
    node=dict(pad=18, thickness=18, label=labels, color=colors),
    link=dict(
        source=list(range(z)) + [z] * len(outflows),
        target=[z] * z + list(range(z + 1, len(labels))),
        value=[v for _, v in inflows] + [v for _, v in outflows],
        color=[colors[i] for i in range(z)]
              + [colors[z + 1 + i] for i in range(len(outflows))],
    ),
))
fig.update_layout(title="Apt 305 — ISO 52016-1 annual energy balance (kWh/yr)",
                  height=560)
fig.show()

In [ ]:
# 11. THE CHECK: both harnesses must report the same baseline engine result.
#
# This is what the two charts could not settle on their own. It compares the
# weather each run actually used (from run_meta.json) BEFORE comparing numbers,
# so a weather mismatch is reported as a weather mismatch, not as a physics bug.
!python examples/check_baseline_consistency.py

## Reading the two charts together

The `Baseline` column of cell 5 and the `ISO 52016-1` column of cell 9 are the **same
engine on the same building**, so they must be identical — cell 11 asserts it to 1e-6.
If they ever diverge again, check the recorded weather before suspecting the physics:

```
results/apt305/run_meta.json
results/baseline_vs_ep/run_meta.json
```

## Varying the building

`examples/apt305_building.py` holds the building dictionary and imports no engine, so the
same definition feeds every engine version. Edit it and re-run cells 4 and 8.

## Using different weather

Drop another `.epw` into `weather_cache/`, re-run cell 3, then cells 4 and 8. The file is
validated against the building's coordinates: beyond 2.5° it is rejected outright, and
between 1.5° and 2.5° it is accepted with the station named on every chart. Add
`--allow-site-mismatch` to override — results are then for the EPW's own location.

## Isolating a single change without switching branches

Both changes are on by default on the top branch and can be switched off individually:

| Option | Effect |
|---|---|
| `dynamic_window_properties=False` | Recovers exact baseline behaviour |
| `dynamic_surface_heat_transfer=False` | Recovers change-1-only behaviour |
| `window_angular_solar_model='none'` | Disables the Karlsson–Roos angular correction |

Both are verified inert when disabled: every compared metric matches the reference engine
exactly, at zero tolerance.